In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: /Users/yangjaehoon/Desktop/StockLens
RAW_DIR: /Users/yangjaehoon/Desktop/StockLens/data/raw
PROCESSED_DIR: /Users/yangjaehoon/Desktop/StockLens/data/processed


In [3]:
for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

data/raw/.DS_Store
data/raw/.gitkeep
data/raw/kiwoom/.DS_Store
data/raw/kiwoom/ka10081/.DS_Store
data/raw/kiwoom/ka10081/005380/20260831T051621327442Z.json
data/raw/kiwoom/ka10081/005380/20260831T032633950419Z.json
data/raw/kiwoom/ka10081/005380/20260831T032705972234Z.json
data/raw/kiwoom/ka10081/005380/20260901T023509357887Z.json
data/raw/kiwoom/ka10081/035720/20260831T032635079878Z.json
data/raw/kiwoom/ka10081/035720/20260831T051621473616Z.json
data/raw/kiwoom/ka10081/035720/20260831T032706052542Z.json
data/raw/kiwoom/ka10081/035720/20260901T023509550392Z.json
data/raw/kiwoom/ka10081/035420/20260831T032706011884Z.json
data/raw/kiwoom/ka10081/035420/20260831T051621394346Z.json
data/raw/kiwoom/ka10081/035420/20260901T023509461123Z.json
data/raw/kiwoom/ka10081/035420/20260831T032633987564Z.json
data/raw/kiwoom/ka10081/000660/20260901T023509303108Z.json
data/raw/kiwoom/ka10081/000660/20260831T032633915363Z.json
data/raw/kiwoom/ka10081/000660/20260831T032705929853Z.json
data/raw/kiwoom/ka

In [4]:
import json

sample_file = next(
    (RAW_DIR / "kiwoom" / "ka10081" / "005930").glob("*.json")
)

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print(type(sample))
print(sample.keys() if isinstance(sample, dict) else "not a dict")

<class 'dict'>
dict_keys(['stk_cd', 'stk_dt_pole_chart_qry', 'return_code', 'return_msg'])


In [5]:
chart_data = sample["stk_dt_pole_chart_qry"]

print(type(chart_data))
print("rows:", len(chart_data))
print("first row:")
print(chart_data[0])

<class 'list'>
rows: 600
first row:
{'cur_prc': '253750', 'trde_qty': '8308403', 'trde_prica': '2080524', 'dt': '20260831', 'open_pric': '249000', 'high_pric': '255000', 'low_pric': '246000', 'pred_pre': '-3250', 'pred_pre_sig': '5', 'trde_tern_rt': '+0.14'}


In [8]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import engineering
from src.features import target

print("src import OK")
print("engineering:", dir(engineering))
print("target:", dir(target))

src import OK
engineering: ['DailyBar', 'FEATURE_COLUMNS', 'Sequence', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'build_features', 'np', 'pd']
target: ['DailyBar', 'Sequence', 'TARGET_COLUMN', 'TARGET_HORIZON', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'build_target', 'pd']


In [9]:
import inspect

print("build_features:")
print(inspect.signature(engineering.build_features))

print("\nbuild_target:")
print(inspect.signature(target.build_target))

print("\nFEATURE_COLUMNS:")
print(engineering.FEATURE_COLUMNS)

print("\nTARGET_COLUMN:")
print(target.TARGET_COLUMN)

print("TARGET_HORIZON:")
print(target.TARGET_HORIZON)

build_features:
(bars: 'Sequence[DailyBar]') -> 'pd.DataFrame'

build_target:
(bars: 'Sequence[DailyBar]', *, horizon: 'int' = 5) -> 'pd.DataFrame'

FEATURE_COLUMNS:
('return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20')

TARGET_COLUMN:
target_return_5d
TARGET_HORIZON:
5


In [10]:
from src.data.dataset import DailyBar

print(DailyBar)

<class 'src.data.models.DailyBar'>


In [11]:
import json

def load_daily_bars(stock_code: str):
    stock_dir = RAW_DIR / "kiwoom" / "ka10081" / stock_code
    files = sorted(stock_dir.glob("*.json"))

    if not files:
        raise FileNotFoundError(f"No JSON files found: {stock_dir}")

    rows = []

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows.extend(data["stk_dt_pole_chart_qry"])

    # 같은 날짜가 여러 JSON에 있을 수 있으므로 날짜 기준 중복 제거
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset="dt").sort_values("dt").reset_index(drop=True)

    print(stock_code, ":", len(df), "bars")
    print(df["dt"].min(), "~", df["dt"].max())

    return df

In [12]:
raw_df = load_daily_bars("005930")

raw_df.head()

005930 : 601 bars
20240313 ~ 20260901


,cur_prc,trde_qty,trde_prica,dt,open_pric,high_pric,low_pric,pred_pre,pred_pre_sig,trde_tern_rt
0,74100,15243134,1126213,20240313,73700,74100,73500,+800,2,+0.26
1,74300,22545539,1672984,20240314,74400,74500,73600,+200,2,+0.38
2,72300,22580555,1646273,20240315,73400,73700,72300,-2000,5,+0.38
3,72800,11520348,837896,20240318,72600,73000,72500,+500,2,+0.19
4,72800,15376066,1109067,20240319,72300,73000,71700,0,3,+0.26


In [13]:
import inspect

print(inspect.signature(DailyBar))

(stock_code: 'str', trade_date: 'date', open_price: 'int', high_price: 'int', low_price: 'int', close_price: 'int', volume: 'int', trade_value_million_krw: 'int', previous_close_change: 'int', previous_close_change_sign: 'int', turnover_rate: 'Decimal') -> None


In [14]:
from datetime import date
from decimal import Decimal

def to_daily_bars(df: pd.DataFrame, stock_code: str) -> list[DailyBar]:
    bars = []

    for row in df.itertuples(index=False):
        bars.append(
            DailyBar(
                stock_code=stock_code,
                trade_date=date(
                    int(str(row.dt)[:4]),
                    int(str(row.dt)[4:6]),
                    int(str(row.dt)[6:8]),
                ),
                open_price=int(row.open_pric),
                high_price=int(row.high_pric),
                low_price=int(row.low_pric),
                close_price=int(row.cur_prc),
                volume=int(row.trde_qty),
                trade_value_million_krw=int(row.trde_prica),
                previous_close_change=int(row.pred_pre),
                previous_close_change_sign=int(row.pred_pre_sig),
                turnover_rate=Decimal(str(row.trde_tern_rt)),
            )
        )

    return bars

In [15]:
bars = to_daily_bars(raw_df, "005930")

print("bars:", len(bars))
print(bars[0])

bars: 601
DailyBar(stock_code='005930', trade_date=datetime.date(2024, 3, 13), open_price=73700, high_price=74100, low_price=73500, close_price=74100, volume=15243134, trade_value_million_krw=1126213, previous_close_change=800, previous_close_change_sign=2, turnover_rate=Decimal('0.26'))


In [16]:
features_df = engineering.build_features(bars)

print("shape:", features_df.shape)
print("columns:")
print(features_df.columns.tolist())

features_df.head()

shape: (601, 26)
columns:
['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,roc_20,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20
0,2024-03-13,NaN,NaN,NaN,NaN,0.005427,0.008163,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-14,0.002699,NaN,NaN,NaN,-0.001344,0.012228,0.004049,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.479062,NaN,NaN
2,2024-03-15,-0.026918,NaN,NaN,NaN,-0.014986,0.019364,-0.012113,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001553,NaN,NaN
3,2024-03-18,0.006916,NaN,NaN,NaN,0.002755,0.006897,0.004149,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.489811,NaN,NaN
4,2024-03-19,0.000000,NaN,NaN,NaN,0.006916,0.018131,-0.006868,73260.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.334688,NaN,NaN


In [17]:
target_df = target.build_target(
    bars,
    horizon=target.TARGET_HORIZON,
)

print("shape:", target_df.shape)
print("columns:", target_df.columns.tolist())

target_df.head(10)

shape: (601, 3)
columns: ['stock_code', 'trade_date', 'target_return_5d']


,stock_code,trade_date,target_return_5d
0,005930,2024-03-13,0.037787
1,005930,2024-03-14,0.067295
2,005930,2024-03-15,0.091286
3,005930,2024-03-18,0.074176
4,005930,2024-03-19,0.097527
5,005930,2024-03-20,0.037711
6,005930,2024-03-21,0.018916
7,005930,2024-03-22,0.044360
8,005930,2024-03-25,0.048593
9,005930,2024-03-26,0.063830


In [18]:
prepared_df = features_df.merge(
    target_df,
    on="trade_date",
    how="inner",
)

print("shape:", prepared_df.shape)
print("columns:", prepared_df.columns.tolist())

prepared_df.head()

shape: (601, 28)
columns: ['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'stock_code', 'target_return_5d']


,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,stock_code,target_return_5d
0,2024-03-13,NaN,NaN,NaN,NaN,0.005427,0.008163,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005930,0.037787
1,2024-03-14,0.002699,NaN,NaN,NaN,-0.001344,0.012228,0.004049,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.479062,NaN,NaN,005930,0.067295
2,2024-03-15,-0.026918,NaN,NaN,NaN,-0.014986,0.019364,-0.012113,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.001553,NaN,NaN,005930,0.091286
3,2024-03-18,0.006916,NaN,NaN,NaN,0.002755,0.006897,0.004149,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.489811,NaN,NaN,005930,0.074176
4,2024-03-19,0.000000,NaN,NaN,NaN,0.006916,0.018131,-0.006868,73260.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.334688,NaN,NaN,005930,0.097527


In [19]:
missing = prepared_df.isna().sum()

missing[missing > 0]

return_1d            1
return_5d            5
return_10d          10
return_20d          20
gap                  1
sma_5                4
sma_20              19
sma_60              59
price_to_sma_5       4
price_to_sma_20     19
price_to_sma_60     59
rsi_14              14
roc_10              10
roc_20              20
macd                25
macd_signal         33
macd_hist           33
volatility_5         5
volatility_20       20
atr_14              13
volume_change_1d     1
volume_sma_20       19
volume_ratio_20     19
target_return_5d     5
dtype: int64

In [20]:
print("전체 행:", len(prepared_df))
print("결측치가 하나라도 있는 행:", prepared_df.isna().any(axis=1).sum())

전체 행: 601
결측치가 하나라도 있는 행: 64


In [21]:
clean_df = prepared_df.dropna(
    subset=list(engineering.FEATURE_COLUMNS) + [target.TARGET_COLUMN]
).reset_index(drop=True)

print("Before:", prepared_df.shape)
print("After :", clean_df.shape)
print("Removed:", len(prepared_df) - len(clean_df))

print("\nRemaining missing values:")
print(clean_df.isna().sum().sum())

Before: (601, 28)
After : (537, 28)
Removed: 64

Remaining missing values:
0


In [22]:
print("첫 70행의 결측치:")
print(
    prepared_df[list(engineering.FEATURE_COLUMNS) + [target.TARGET_COLUMN]]
    .isna()
    .sum(axis=1)
    .head(70)
)

첫 70행의 결측치:
0     23
1     20
2     20
3     20
4     18
      ..
65     0
66     0
67     0
68     0
69     0
Length: 70, dtype: int64


In [23]:
print("shape:", clean_df.shape)
print("missing:", clean_df.isna().sum().sum())

print("\nDate range:")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

shape: (537, 28)
missing: 0

Date range:
2024-06-11 ~ 2026-08-25


In [24]:
print("=== Dataset columns ===")
print(clean_df.columns.tolist())

print("\n=== Date range ===")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

print("\n=== Target range ===")
print(clean_df["target_return_5d"].min(), "~", clean_df["target_return_5d"].max())

=== Dataset columns ===
['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'stock_code', 'target_return_5d']

=== Date range ===
2024-06-11 ~ 2026-08-25

=== Target range ===
-0.23333333333333328 ~ 0.29478458049886624


In [25]:
clean_df.tail(10)[
    ["trade_date", "target_return_5d"]
]

,trade_date,target_return_5d
527,2026-08-11,0.033403
528,2026-08-12,0.060665
529,2026-08-13,0.050373
530,2026-08-14,-0.063752
531,2026-08-18,-0.042831
532,2026-08-19,0.056566
533,2026-08-20,-0.018450
534,2026-08-21,-0.087034
535,2026-08-24,-0.019455
536,2026-08-25,0.006809


In [26]:
# Feature가 현재 시점 이후의 데이터를 사용하지 않는지 확인하기 위한
# 가장 기본적인 시점 검증

print("Feature columns:")
print(list(engineering.FEATURE_COLUMNS))

print("\nClean dataset:")
print(clean_df.shape)

print("\nDate range:")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

Feature columns:
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']

Clean dataset:
(537, 28)

Date range:
2024-06-11 ~ 2026-08-25


In [29]:
final_columns = [
    "stock_code",
    "trade_date",
    *engineering.FEATURE_COLUMNS,
    target.TARGET_COLUMN,
]

final_df = clean_df[final_columns].copy()

print(final_df.shape)
print(final_df.columns.tolist())

(537, 28)
['stock_code', 'trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [28]:
STOCK_CODES = [
    "000660",
    "005380",
    "005930",
    "035420",
    "035720",
]

print(STOCK_CODES)

['000660', '005380', '005930', '035420', '035720']


In [30]:
import json
from datetime import date
from decimal import Decimal

from src.data.models import DailyBar


def load_bars(stock_code):
    stock_dir = RAW_DIR / "kiwoom" / "ka10081" / stock_code

    rows_by_date = {}

    for json_path in sorted(stock_dir.glob("*.json")):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows = data.get("stk_dt_pole_chart_qry", [])

        for row in rows:
            dt = row["dt"]

            rows_by_date[dt] = DailyBar(
                stock_code=stock_code,
                trade_date=date(
                    int(dt[:4]),
                    int(dt[4:6]),
                    int(dt[6:8]),
                ),
                open_price=int(row["open_pric"]),
                high_price=int(row["high_pric"]),
                low_price=int(row["low_pric"]),
                close_price=int(row["cur_prc"]),
                volume=int(row["trde_qty"]),
                trade_value_million_krw=int(row["trde_prica"]),
                previous_close_change=int(row["pred_pre"]),
                previous_close_change_sign=int(row["pred_pre_sig"]),
                turnover_rate=Decimal(row["trde_tern_rt"]),
            )

    return sorted(rows_by_date.values(), key=lambda x: x.trade_date)

In [31]:
prepared_dfs = []

for stock_code in STOCK_CODES:
    bars = load_bars(stock_code)

    features = engineering.build_features(bars)
    targets = target.build_target(
        bars,
        horizon=target.TARGET_HORIZON,
    )

    df = features.merge(
        targets,
        on="trade_date",
        how="inner",
    )

    df["stock_code"] = stock_code

    df = df[
        [
            "stock_code",
            "trade_date",
            *engineering.FEATURE_COLUMNS,
            target.TARGET_COLUMN,
        ]
    ]

    df = df.dropna().reset_index(drop=True)

    prepared_dfs.append(df)

    print(
        stock_code,
        "bars:", len(bars),
        "prepared:", df.shape,
    )

prepared_df = pd.concat(
    prepared_dfs,
    ignore_index=True,
)

print("\nTOTAL:", prepared_df.shape)

000660 bars: 601 prepared: (537, 28)
005380 bars: 601 prepared: (537, 28)
005930 bars: 601 prepared: (537, 28)
035420 bars: 601 prepared: (537, 28)
035720 bars: 601 prepared: (537, 28)

TOTAL: (2685, 28)


In [32]:
output_path = PROCESSED_DIR / "ml_dataset.csv"

prepared_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", prepared_df.shape)

Saved: /Users/yangjaehoon/Desktop/StockLens/data/processed/ml_dataset.csv
Shape: (2685, 28)
